# Comparar regret + VND com força bruta
Use o kernel **Python (TrabIC)**. Este notebook chama os mesmos executáveis da CLI; não duplica os algoritmos nem altera as instâncias. A força bruta serve apenas para exemplos pequenos. Só o status `otimo` certifica o custo ótimo.

In [ ]:
from pathlib import Path
import sys, subprocess, json
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'experimentos.py').is_file() and (p / 'main.py').is_file()), None)
if raiz is None:
    raise RuntimeError('Abra o notebook dentro do repositório TrabIC.')
instancia = raiz / 'data/exemplos/pequena.txt'
tipos_por_local = 2
seed = 42
print('Python:', sys.executable)
print('Instância:', instancia)

In [ ]:
def executar(script, argumentos):
    processo = subprocess.run([sys.executable, str(raiz / script), '--instancia', str(instancia), '--tipos-por-local', str(tipos_por_local), *argumentos], cwd=raiz, capture_output=True, text=True, timeout=120)
    if not processo.stdout.strip():
        raise RuntimeError(processo.stderr or f'Execução terminou com código {processo.returncode}')
    resultado = json.loads(processo.stdout)
    print('Código de saída:', processo.returncode)
    return resultado
heuristica = executar('main.py', ['--seed', str(seed), '--execucoes', '10'])
print(json.dumps(heuristica, indent=2, ensure_ascii=False))

In [ ]:
exato = executar('brute_force.py', ['--tempo-limite', '10'])
print(json.dumps(exato, indent=2, ensure_ascii=False))

In [ ]:
custos = [r['custo_final'] for r in heuristica['execucoes'] if r['status'] == 'viavel']
if exato['status'] == 'otimo' and custos:
    melhor = min(custos)
    diferenca = melhor - exato['custo']
    gap_percentual = 100 * diferenca / exato['custo'] if exato['custo'] > 0 else (0.0 if melhor == 0 else None)
    print({'melhor_heuristica': melhor, 'otimo': exato['custo'], 'diferenca': diferenca, 'gap_percentual': gap_percentual, 'unidade': exato['unidade']})
else:
    print('Sem comparação de otimalidade: exato interrompido/inviável ou nenhuma solução heurística viável.')